# Análise de experimentos

Antes: `cmake --build build` e `pip install -r requirements.txt` (instala o pacote `tsp` de `scripts/`).

Cada experimento é uma pasta em `experiments/` com um `experiment.json`. `experiment.run()` só roda
as execuções que faltam, então rodar este notebook de novo é barato. Para rodar fora do notebook:
`python scripts/run_experiment.py experiments/operators`.

In [ ]:
from tsp import compare, experiment, load, plots

## Rodar

Execuções já concluídas com os mesmos argumentos são puladas.

In [ ]:
exp = experiment.run("operators", jobs=8)
df = load.summary(exp)
load.factors(exp)

## Tabela por configuração

Ordenada pela mediana do comprimento final (menor é melhor), sobre todas as seeds e mapas.

In [ ]:
compare.table(df).round(1)

## Convergência

Mediana entre execuções a cada época, com faixa interquartil.

In [ ]:
hist = load.history(exp)
fig = plots.convergence(hist, hue="mutation", col="crossover")

## Resultado final

In [ ]:
fig = plots.box(df, x="mutation", hue="crossover")

In [ ]:
fig = plots.heatmap(df, rows="mutation", cols="crossover")

## A diferença é real?

Pareado por mapa e seed. Friedman diz se algum nível difere; o par a par (Wilcoxon + Holm) diz quais.

In [ ]:
f = compare.friedman(df, "mutation")
print(f"p = {f['p_value']:.2g}")
f["mean_rank"]

In [ ]:
compare.pairwise(df, "mutation").round(4)

## Rotas

In [ ]:
best = df.loc[df["best"].idxmin()]
fig = plots.route_evolution(exp.dir / "runs" / best["run"])

## Variações e comparação entre experimentos

Um experimento pode ser definido aqui mesmo; ele é gravado em `experiments/<nome>/experiment.json`.
Como cada execução guarda a própria config, dá para juntar experimentos diferentes numa tabela só.

In [ ]:
# experiment.run("pop_rapido", base={"n": 50, "cross": "ox", "mutation": "inversion"},
#                grid={"pop": [100, 400]}, seeds=10)

both = load.summary("operators", "rates")
compare.table(both, by=["experiment", "mutation"]).round(1)

## Salvar para o relatório

In [ ]:
fig = plots.box(df, x="mutation", hue="crossover")
plots.save(fig, exp, "box_mutation")  # experiments/operators/figures/box_mutation.png